### Jupyter notebook to get the expanded organ masks, whole body masks, head masks, postprocessed tissue masks for quantification
#### What do you need:
##### 1. Downsampled raw image, saved as nii.gz file
##### 2. Organ mask from the Tissue Module for the downsampled raw image, saved as nii.gz file
##### 3. Tissue mask from the Tissue Module which is downsampled to the same resolution as the above organ mask

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import cv2
import scipy
import nibabel as nib
from skimage.segmentation import expand_labels

In [ ]:
# For overlay visualization
from ipywidgets import interact, IntSlider
def show_slice(raw, mask, slice_idx):
    plt.figure(figsize=(12, 6))
    plt.imshow(raw[:, :, slice_idx], cmap='gray', vmin=0, vmax=2000) #, vmin=0, vmax=1
    plt.imshow(mask[:, :, slice_idx], cmap="Reds",alpha=0.4)  # overlay mask with transparency
    plt.axis('off')
    plt.title(f"Slice {slice_idx}")
    plt.show()

In [ ]:
organ_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organmask_xy40z10.nii.gz"
raw_image_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/C00_xy40z10.nii.gz"
tissue_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/tissuemask_xy40z10.nii.gz"

In [ ]:
## check the sizes of different masks and raw image to be same
raw_image= nib.load(raw_image_path).get_fdata()
organ_mask = nib.load(organ_mask_path).get_fdata()
tissue_mask = nib.load(tissue_mask_path).get_fdata()


if raw_image.shape != organ_mask.shape:
    raise ValueError(f"Shape mismatch between raw image and organ mask: {raw_image.shape} != {organ_mask.shape}")
if raw_image.shape != tissue_mask.shape:
    raise ValueError(f"Shape mismatch between raw image and tissue mask: {raw_image.shape} != {tissue_mask.shape}")

In [ ]:
## define the path for saving expanded organ mask, whole body mask and postprocessed tissue mask
organ_mask_grow_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organmask_grow_xy40z10.nii.gz"
wb_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/wholebodymask_xy40z10.nii.gz"
tissue_mask_post_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/tissuemask_post_xy40z10.nii.gz"
head_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/headmask_xy40z10.nii.gz"

### Grow organ mask for quantification

In [ ]:
def grow_organ_mask(np_organ_mask, distance=8):

    # grow the mask for organs without brain
    np_organ_mask[np_organ_mask==5]=0
    np_organ_mask = expand_labels(np_organ_mask, distance=distance) 
    return np_organ_mask
organ_mask_grow = grow_organ_mask(organ_mask)

In [ ]:
# Visualize the organ mask by overlaying it on the raw image
interact(
    lambda slice_idx: show_slice(raw_image, organ_mask_grow, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(organ_mask_grow, affine=affmat)
nib.save(NiftiObject, organ_mask_grow_path)

### Create wholebody mask

In [ ]:
init_wbmask  = np.zeros_like(raw_image)
init_wbmask[raw_image>160]=1 # By thresholding first to seperate background and foreground


In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, init_wbmask, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=250), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
def get_wb_mask(arr_thresh, arr_organmask):
    arr_wb = arr_thresh + arr_organmask
    arr_wb[arr_wb>0]=1
    return arr_wb

wbmask = get_wb_mask(init_wbmask, organ_mask)

In [ ]:
def dilate_vol(vol, dilate_iter):
    vol_dilate = scipy.ndimage.binary_dilation(vol, iterations = dilate_iter)
    return vol_dilate

def erode_vol(vol, erode_iter):
    vol_erode = scipy.ndimage.binary_erosion(vol, iterations = erode_iter)
    return vol_erode

wbmask_pro = dilate_vol(erode_vol(wbmask, 2), 8)  # with dilate and erode operation, remove noise outside mouse bady and ensure the mask covers whole body

In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, wbmask_pro, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=250), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(wbmask_pro.astype(np.uint8), affine=affmat)
nib.save(NiftiObject,wb_mask_path)

## Postprocessing tissue mask with wholebody mask and organ mask

In [ ]:
## Mask out predictions outside whole body region based on whole body mask
## Then mask out predictions inside organs
wbmask = nib.load(wb_mask_path).get_fdata()
tissue_mask_post = tissue_mask * wbmask
tissue_mask_post = tissue_mask_post* (organ_mask==0)

In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, tissue_mask_post, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=203), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(tissue_mask_post.astype(np.uint8), affine=affmat)
nib.save(NiftiObject, tissue_mask_post_path)

[0. 1. 2. 3. 4.]


### Create head mask

In [28]:
raw_thresh = np.zeros_like(raw_image)
raw_thresh[raw_image>180]=1
raw_thresh.shape

(511, 1110, 210)

In [ ]:
brain_mask = np.zeros_like(organ_mask)
brain_mask[organ_mask==5]=1
structure = np.ones((3, 3, 6), dtype=int)
brain_mask = scipy.ndimage.binary_dilation(brain_mask, structure, iterations=50).astype(np.uint8)

[0. 1.]


In [31]:
brain_mask.shape

(511, 1110, 210)

In [32]:
merge_head_mask = raw_thresh * brain_mask
merge_head_mask.shape

(511, 1110, 210)

In [33]:
interact(
    lambda slice_idx: show_slice(raw_image, merge_head_mask, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [34]:
merge_head_mask_fill = scipy.ndimage.binary_dilation(scipy.ndimage.binary_fill_holes(merge_head_mask, axes = 2), iterations=5).astype(np.uint8)

In [35]:
interact(
    lambda slice_idx: show_slice(raw_image, merge_head_mask_fill, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

#### Note: the head mask generated here may still need further manual revision.

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(merge_head_mask_fill, affine=affmat)
nib.save(NiftiObject, head_mask_path)